In [23]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

MODEL_FILE="model.pkl"
PIPELINE_FILE="pipline.pkl"

def build_pipline(num_data,label_data):
    lean_pipline=Pipeline([
        ("impute",SimpleImputer(strategy="mean")),
        ("standrized",StandardScaler())
    ])
    cat_pipline=Pipeline([
        ("ocean_total",OneHotEncoder(handle_unknown="ignore"))
    ])
    full_pipline=ColumnTransformer([
        ("num",lean_pipline,num_data),
        ("cat",cat_pipline,label_data)
    ])
    return full_pipline

    
if not os.path.exists(MODEL_FILE):
    data = pd.read_csv("cali-data.csv")
    data["income_cat"]=pd.cut(data["median_income"] , bins=[0,1.5,2.5,3.5,4.5,6,np.inf],labels=[1,2,3,4,5,6])
    
    split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
    
    for train_index, test_index in split.split(data,data["income_cat"]):
        strat_train=data.loc[train_index].drop("income_cat",axis=1)
        data.loc[test_index].drop("income_cat",axis=1).to_csv("dataframe.csv",index=False)
        
    datatr=strat_train.copy()
    data_labels=datatr["median_house_value"].copy()
    datatr_feature=datatr.drop("median_house_value",axis=1)
     
     # here no we can seprate the table into two parts in number and labels
    num_data=datatr_feature.drop("ocean_proximity",axis=1).columns.tolist()
    label_data=["ocean_proximity"]
    
    pipline=build_pipline(num_data,label_data)
    house_prepared=pipline.fit_transform(datatr)
    
    model=RandomForestRegressor(random_state=42)
    model.fit(house_prepared,data_labels)
    
    
    decision=RandomForestRegressor(random_state=42)
    out=decision.fit(house_prepared,data_labels)
    # max_samples aur min_samples_leaf badalne se model ka size 30% tak kam ho jata hai
    # model = RandomForestRegressor(
    #     max_samples=0.8,      # Har tree ko 80% data milega (size kam hoga)
    #     min_samples_leaf=2,   # Unnecessary branches nahi banengi
    #     random_state=42,
    #     n_jobs=-1             # Isse training fast hogi
    # )
    
    # decision = RandomForestRegressor(
    #     max_samples=0.8,
    #     min_samples_leaf=2,
    #     random_state=42,
    #     n_jobs=-1
    # )
    # forest_pre=out.predict(house_prepared)

    scoring_random=cross_val_score(
    decision,house_prepared,data_labels,cv=10,scoring="neg_mean_squared_error")
    print(f"The tree score is : {scoring_random}")
    print(pd.Series(scoring_random).describe())
    # Accuracy data ko ek dictionary mein daalein
    
    rmse_scores = np.sqrt(-scoring_random)
    accuracy_info = {
        "mean_rmse": float(rmse_scores.mean()),
        "std_rmse": float(rmse_scores.std())
    }
    
    # Ise ek alag file mein save kar dein
    joblib.dump(accuracy_info, "accuracy.pkl")
    print(f"Real Accuracy Saved! Mean: {accuracy_info['mean_rmse']:.2f}, Std: {accuracy_info['std_rmse']:.2f}")
        
    
    joblib.dump(model,MODEL_FILE,compress=('lzma',9))
    joblib.dump(pipline,PIPELINE_FILE)
    print("model train and saved")
else:
    model=joblib.load(MODEL_FILE)
    pipline=joblib.load(PIPELINE_FILE)
    
    input_data=pd.read_csv("dataframe.csv")
    transform_data=pipline.transform(input_data)
    predection=model.predict(transform_data)
    
    input_data["median_price"]=predection
    input_data.to_csv("output_result.csv",index=False)
    print("work is complete")
    
    
    

work is complete
